In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- 1) Setup headless Chrome (requires selenium>=4.6 for auto-driver) ---
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")

driver = webdriver.Chrome(options=chrome_options)

try:
    # --- 2) Extract all customer story links via pagination ---
    search_url = (
       "https://www.microsoft.com/de-de/customers/search/?filters=language%3Agerman%2Clanguage%3Aenglish%2Cindustry%3Amanufacturing%2Cindustry%3Aautomotive&sortBy=PublishedDate+Desc"
    )
    driver.get(search_url)
    time.sleep(5)  # let initial content load

    all_links = []
    js_code = """
    var elements = document.querySelectorAll("a.btn.btn--md");
    var links = [];
    elements.forEach(function(element) {
        var href = element.getAttribute("href");
        if (href && href !== '#') {
            links.push(href);
        }
    });
    return links;
    """
    while True:
        # grab links on this page
        for href in driver.execute_script(js_code):
            all_links.add(href)

        # try to click “next”
        try:
            nxt = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "#right-arrow button"))
            )
            parent_cls = nxt.find_element(By.XPATH, "..").get_attribute("class")
            if "disabled" in parent_cls:
                break
            nxt.click()
            time.sleep(5)
        except:
            break

    print(f"Found {len(all_links)} story links.")

    # --- 3) Visit each and scrape the fields + your target div ---
    results = []
    for link in all_links:
        if not link.startswith("http"):
            print("Skipping unsupported link:", link)
            continue
        try:
            driver.get(link)
        except Exception as e:
            print("Failed to load", link, "skipping:", e)
            continue

        driver.get(link)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR,
             "#project-details-section"))
        )

        date_element = driver.execute_script("""
          let e = document.querySelector(
            "#section-master-customer-share-dialog .block-heading__eyebrow > div"
          );
          return e ? e.innerText.trim() : null;
        """)
        company_name = driver.execute_script("""
          let e = document.querySelector(
            "#project-details-section .project-details-bar__body ul li"
          );
          return e ? e.innerText.trim() : null;
        """)
        related_products = driver.execute_script("""
          return Array.from(
            document.querySelectorAll(
              "#project-details-section .project-details-bar__body ul:nth-of-type(2) li"
            )
          ).map(li => li.innerText.trim());
        """)
        business_need = driver.execute_script("""
          let e = document.querySelector(
            "#project-details-section .project-details-bar__body ul:nth-of-type(3) li"
          );
          return e ? e.innerText.trim() : null;
        """)
        industry = driver.execute_script("""
          let e = document.querySelector(
            "#project-details-section .project-details-bar__body ul:nth-of-type(4) li"
          );
          return e ? e.innerText.trim() : null;
        """)
        employees = driver.execute_script("""
          let e = document.querySelector(
            "#project-details-section .project-details-bar__body ul:nth-of-type(5) li"
          );
          return e ? e.innerText.trim() : null;
        """)
        country = driver.execute_script("""
          let e = document.querySelector(
            "#project-details-section .project-details-bar__body ul:nth-of-type(6) li"
          );
          return e ? e.innerText.trim() : null;
        """)
        # your custom selector:
        topic = driver.execute_script("""
          let e = document.querySelector(
            "#chapter-oce009 > div > div > div.layout.layout--cols-2.layout--editorial "
            + "> article > div > div > div > div > div"
          );
          return e ? e.innerText.trim() : null;
        """)

        results.append({
            "url": link,
            "date": date_element,
            "company_name": company_name,
            "related_products_count": len(related_products),
            "related_products": related_products,
            "business_need": business_need,
            "industry": industry,
            "employees": employees,
            "country": country,
            "topic": topic
        })
        print("Scraped:", link)

finally:
    driver.quit()

# --- 4) Build DataFrame & Save ---
df = pd.DataFrame(results)
print(df)
df.to_csv("1_scraped_microsoft_customers.csv", index=False)


AttributeError: 'list' object has no attribute 'add'

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set up Selenium with headless Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")

# Replace with the actual path to your ChromeDriver executable
driver_path = "/opt/homebrew/bin/chromedriver"
service = Service(driver_path)
driver = webdriver.Chrome(options=chrome_options)

### PART 1: Extract All Links from the Search Pages

search_url = ("https://www.microsoft.com/de-de/customers/search/?filters=language%3Agerman%2Clanguage%3Aenglish%2Cindustry%3Amanufacturing%2Cindustry%3Aautomotive&sortBy=PublishedDate+Desc")
driver.get(search_url)
time.sleep(10)  # Wait for the dynamic content to load

all_links = []
js_code = """
var elements = document.querySelectorAll("a.btn.btn--md");
var links = [];
elements.forEach(function(element) {
    var href = element.getAttribute("href");
    if (href && href !== '#') {
        links.push(href);
    }
});
return links;
"""

# Loop through pagination
while True:
    # Execute JS to get the links on the current page
    extracted_links = driver.execute_script(js_code)
    all_links.extend(extracted_links)
    print(f"Links from current page: {extracted_links}")
    
    try:
        # Wait for the "next page" button to be clickable
        next_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "#right-arrow button"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
        
        # Check if the "next" button's parent has a disabled class (indicating the last page)
        next_button_parent = next_button.find_element(By.XPATH, "..")
        if 'disabled' in next_button_parent.get_attribute("class"):
            print("Reached the last page. Exiting pagination loop.")
            break
        
        # Click the "next" button
        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(10)  # Adjust sleep as needed for the next page to load
        
    except Exception as e:
        print(f"Error during pagination: {e}")
        break

# Remove duplicate links (if any)
all_links = list(set(all_links))
print("Total Extracted Links:", len(all_links))

### PART 2: For Each Extracted Link, Scrape the Required Information

results = []

for link in all_links:
    if not link.startswith("http"):
        print("Skipping unsupported link:", link)
        continue
    try:
      driver.get(link)
      print(f"Scraping data from: {link}")
      time.sleep(8)  # Allow time for page load
    except Exception as e:
        print("Failed to load", link, "skipping:", e)
        continue
    
    # Extract the various elements using execute_script with your provided JS paths.
    date_element = driver.execute_script("""
      var elem = document.querySelector("#section-master-customer-share-dialog > div:nth-child(1) > div > div > div > div > div > div.block-heading__eyebrow > div");
      return elem ? elem.innerText.trim() : null;
    """)
    
    company_name = driver.execute_script("""
      var elem = document.querySelector("#project-details-section > div > div > div:nth-child(1) > div.project-details-bar__body > div > div:nth-child(1) > div:nth-child(1) > div.parsys > ul > li");
      return elem ? elem.innerText.trim() : null;
    """)
    
    related_products = driver.execute_script("""
      var items = document.querySelectorAll("#project-details-section > div > div > div:nth-child(1) > div.project-details-bar__body > div > div:nth-child(2) > div > div.parsys > ul > li");
      var names = [];
      items.forEach(function(item) {
          names.push(item.innerText.trim());
      });
      return names;
    """)
    
    business_need = driver.execute_script("""
      var elem = document.querySelector("#project-details-section > div > div > div:nth-child(2) > div.project-details-bar__body > div > div:nth-child(2) > div:nth-child(1) > div.parsys > ul > li");
      return elem ? elem.innerText.trim() : null;
    """)
    
    industry = driver.execute_script("""
      var elem = document.querySelector("#project-details-section > div > div > div:nth-child(2) > div.project-details-bar__body > div > div:nth-child(2) > div:nth-child(2) > div.parsys > ul > li");
      return elem ? elem.innerText.trim() : null;
    """)
    
    employees = driver.execute_script("""
      var elem = document.querySelector("#project-details-section > div > div > div:nth-child(2) > div.project-details-bar__body > div > div:nth-child(1) > div:nth-child(1) > div.parsys > ul > li");
      return elem ? elem.innerText.trim() : null;
    """)
    
    country = driver.execute_script("""
      var elem = document.querySelector("#project-details-section > div > div > div:nth-child(2) > div.project-details-bar__body > div > div:nth-child(1) > div:nth-child(2) > div.parsys > ul > li");
      return elem ? elem.innerText.trim() : null;
    """)
    exec_summary = driver.execute_script("""
      var elem = document.querySelector(
      "#exec-summary-section > div > div > div > div"
      );
      return elem ? elem.innerText.trim() : null;
    """)
    
    # Build a dictionary for the current story
    result = {
        "url": link,
        "date": date_element,
        "company_name": company_name,
        "related_products_count": len(related_products),
        "related_products": related_products,
        "business_need": business_need,
        "industry": industry,
        "employees": employees,
        "country": country,
        "exec_summary": exec_summary
    }
    results.append(result)
    print(f"Scraped data from {link}")

# Close the driver once all pages are scraped
driver.quit()

# Convert the results into a DataFrame (expected 84 rows)
df = pd.DataFrame(results)
print("Final DataFrame:")
print(df)

# Optionally, save the DataFrame to a CSV file
df.to_csv("1_scraped_microsoft_customers.csv", index=False)

Links from current page: ['javascript:void(0)', 'https://www.microsoft.com/en/customers/story/23867-glidewell-dental-dynamics-365-finance', 'https://www.microsoft.com/en/customers/story/23716-electroimpact-dynamics-365-finance', 'https://www.microsoft.com/en/customers/story/23680-mediatek-azure-ai-model-catalog', 'https://www.microsoft.com/en/customers/story/23515-gerdau-azure-ai-services', 'https://www.microsoft.com/de-de/customers/story/23482-bucher-industries-ag-azure-open-ai-service', 'https://www.microsoft.com/en/customers/story/23233-bucher-industries-azure-open-ai-service', 'https://www.microsoft.com/en/customers/story/23201-rolls-royce-azure-databricks', 'https://www.microsoft.com/en/customers/story/23047-bridgestone-emea-microsoft-fabric', 'https://www.microsoft.com/en/customers/story/23157-fujifilm-azure', 'https://www.microsoft.com/en/customers/story/23045-bridgestone-emea-microsoft-entra-id', 'https://www.microsoft.com/de-de/customers/story/22436-continental-ag-azure-ai-ser